# NumPy for Machine Learning

### Table of contents
0. [Introduction](#0.-Introduction)
1. [Creating arrays](#1.-Creating-arrays)
2. [Anatomy of an array](#2.-Anatomy-of-an-array)
3. [Indexing, slicing, and views](#3.-Indexing,-slicing,-and-views)
4. [Reshaping, joining, and splitting arrays](#4.-Reshaping,-joining,-and-splitting-arrays)
5. [Iterating arrays](#5.-Iterating-arrays)
6. [Vectorization and broadcasting](#6.-Vectorization-and-broadcasting)
7. [Aggregations, sorting, and searching](#7.-Aggregations,-sorting,-and-searching)
8. [Linear algebra essentials](#8.-Linear-algebra-essentials)
9. [Random number generation](#9.-Random-number-generation)
10. [Inserting, appending, and deleting elements](#10.-Inserting,-appending,-and-deleting-elements)
11. [Performance: why vectorize?](#11.-Performance:-why-vectorize?)
12. [Summary](#12.-Summary)
13. [Exercises](#13.-Exercises)

## 0. Introduction

**NumPy** (*Numerical Python*) is the library every other piece of the Python ML stack sits on top of. Pandas stores numeric columns as NumPy arrays under the hood. Scikit-Learn estimators expect NumPy arrays in and produce them out. PyTorch's `Tensor` API deliberately mirrors NumPy's, so the mental model transfers directly.

**Convention:** NumPy is always imported as `np`. You'll see this in essentially every ML codebase, so stick to it.


In [ ]:
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.5.0


### 0.1 A quick taste: the same operation, two ways

Say you want to double every number in a list of prices. With a plain Python list, `*` does **not** do what you'd hope:


In [ ]:
prices = [10, 20, 30]
print(prices * 2)   # repeats the list, doesn't double each element!

[10, 20, 30, 10, 20, 30]


To actually double each element with plain Python, you need an explicit loop or list comprehension:


In [3]:
doubled = [p * 2 for p in prices]
print(doubled)


[20, 40, 60]


With a NumPy array, `*` does exactly what you'd expect: applied element by element, with no loop written by you.


In [4]:
prices_np = np.array([10, 20, 30])
print(prices_np * 2)


[20 40 60]


### 0.2 Why this is possible: a different memory layout

A Python list is really an array of *pointers*: each element is a separate Python object scattered somewhere in memory, and the list just holds addresses pointing to them. A NumPy array stores the raw numbers back to back, in one contiguous block, all of the same type. That single design choice is what makes whole-array operations like `prices_np * 2` fast: NumPy can walk straight through memory in a tight compiled loop instead of chasing pointers all over the place.

![Python list vs NumPy array memory layout](assets/numpy/numpy-array-vs-list.png)


### 0.3 A first taste of the payoff: memory and speed

We'll measure this properly in Chapter 11, with the full breakdown. For now, here's a quick, honest preview on half a million numbers: summing a NumPy array against summing the equivalent Python list.


In [5]:
import time

n = 5_000_000
py_list = list(range(n))
np_arr = np.arange(n)

start = time.perf_counter()
sum(py_list)
py_time = time.perf_counter() - start

start = time.perf_counter()
np_arr.sum()
np_time = time.perf_counter() - start

print(f"pure Python: {py_time:.4f}s")
print(f"NumPy:       {np_time:.4f}s")
print(f"speedup:     {py_time / np_time:.0f}x")


pure Python: 0.0179s
NumPy:       0.0021s
speedup:     8x


NumPy is both smaller in memory and faster to compute with, for the same numbers. We'll unpack exactly why in Chapter 11. Let's start building the mental model, one piece at a time.


## 1. Creating arrays

We start here, before any theory, because you cannot inspect an array's shape or dtype without one to look at. These constructor functions matter for initializing weights, building test data, and pre-allocating results you'll fill in.


### 1.1 `np.array`: your first array

`np.array()` turns a Python list into an `ndarray`. Notice the type: it's no longer a `list`.


In [6]:
a = np.array([1, 2, 3])
print(a)
print(type(a))


[1 2 3]
<class 'numpy.ndarray'>


### 1.2 Pure Python vs NumPy: building an array of zeros

In plain Python, you'd build a list of zeros like this:


In [7]:
zeros_list = [0] * 5
print(zeros_list)


[0, 0, 0, 0, 0]


NumPy gives you a dedicated function for exactly this, and it generalizes cleanly to more than one dimension, which `[0] * n` cannot do on its own.


In [8]:
zeros_arr = np.zeros(5)
print(zeros_arr)

zeros_2d = np.zeros((2, 3))   # 2 rows, 3 columns
print(zeros_2d)


[0. 0. 0. 0. 0.]
[[0. 0. 0.]
 [0. 0. 0.]]


### 1.3 `np.ones`: pre-allocate an array of ones


In [9]:
print(np.ones((2, 3)))


[[1. 1. 1.]
 [1. 1. 1.]]


### 1.4 `np.full`: pre-allocate with a constant of your choice


In [10]:
print(np.full((2, 3), 7))


[[7 7 7]
 [7 7 7]]


### 1.5 `np.eye`: identity matrix

A square matrix with 1s on the diagonal, 0s elsewhere. Shows up constantly in linear algebra.


In [11]:
print(np.eye(4))


[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


### 1.6 `np.arange`: evenly spaced values by step size

Works like Python's `range()`, but returns an array. `stop` is exclusive.


In [12]:
print(np.arange(0, 10, 2))


[0 2 4 6 8]


### 1.7 `np.linspace`: evenly spaced values by count

`linspace(start, stop, num)` gives you *exactly* `num` points, and `stop` is inclusive by default.


In [13]:
print(np.linspace(0, 1, 5))


[0.   0.25 0.5  0.75 1.  ]


### 1.8 `arange` vs `linspace`, side by side

Same range, different question. `arange` fixes the step; `linspace` fixes the count.


In [14]:
print("arange(0, 1, 0.25):", np.arange(0, 1, 0.25))
print("linspace(0, 1, 4): ", np.linspace(0, 1, 4, endpoint=False))


arange(0, 1, 0.25): [0.   0.25 0.5  0.75]
linspace(0, 1, 4):  [0.   0.25 0.5  0.75]


### 1.9 A quick word of caution: `np.empty`

`np.empty` allocates without writing any values, so it's the fastest way to get an array, but the contents are whatever garbage was already sitting in that memory. Only use it when you're about to overwrite every element yourself.


In [17]:
junk = np.empty((2, 3))
print(junk)   # contents are unpredictable, don't rely on this being zero


[[3.5e-323 3.5e-323 3.5e-323]
 [3.5e-323 3.5e-323 3.5e-323]]


> **ML tie-in:** these constructors are how you initialize weight buffers before training, build an axis of values for a plot, or stub out a result array before filling it in a loop.


## 2. Anatomy of an array

Now that we can make arrays on demand, let's look at what makes an `ndarray` different from a list: the attributes that describe its shape and the type of data it holds.


### 2.1 `.shape`, `.ndim`, `.size`

`np.zeros` gives us an instant way to conjure an array of any shape, so let's use it to read these attributes off directly. `np.zeros((2, 3, 4))` makes an array with 2 blocks, each 3 rows by 4 columns.


In [18]:
c = np.zeros((2, 3, 4))

print("shape:", c.shape)   # one number per axis
print("ndim: ", c.ndim)    # how many axes
print("size: ", c.size)    # total elements: 2 * 3 * 4


shape: (2, 3, 4)
ndim:  3
size:  24


The same attributes on a plain vector and a matrix, so you can see how they scale down:


In [19]:
vector = np.zeros(5)
matrix = np.zeros((2, 3))

print("vector -> shape:", vector.shape, "ndim:", vector.ndim)
print("matrix -> shape:", matrix.shape, "ndim:", matrix.ndim)


vector -> shape: (5,) ndim: 1
matrix -> shape: (2, 3) ndim: 2


### 2.2 `.dtype`: every element shares one type

Unlike a Python list, an `ndarray` is *homogeneously typed*: every element has the same dtype. Integer literals default to `int64`; decimal literals default to `float64`.


In [20]:
ints = np.array([1, 2, 3])
floats = np.array([1.0, 2.0, 3.0])

print(ints.dtype)
print(floats.dtype)


int64
float64


### 2.3 Why dtype matters: memory

You can set the dtype explicitly at creation time. This matters because smaller dtypes use less memory for the exact same numbers, and it's also part of why vectorized operations are fast: NumPy knows the exact size of every element in advance.


In [23]:
f64 = np.zeros(1_000_000, dtype=np.float64)
f32 = np.zeros(1_000_000, dtype=np.float32)
i32 = np.zeros(1_000_000, dtype=np.int32)

print("float64:", f64.nbytes / 1e6, "MB")
print("float32:", f32.nbytes / 1e6, "MB")
print("i32:", i32.nbytes / 1e6, "MB")


float64: 8.0 MB
float32: 4.0 MB
i32: 4.0 MB


### 2.4 The shape gotcha: `(n,)` vs `(n, 1)` vs `(1, n)`

These three hold "the same numbers" but are different shapes, and NumPy (and scikit-learn) treats them differently. This single distinction causes a large share of real-world shape bugs.


![diagram](assets/numpy/diagram_shape_gotcha.png)


In [24]:
vec = np.array([1, 2, 3])            # shape (3,)   - flat, no row/column concept
col = np.array([[1], [2], [3]])      # shape (3, 1) - a column matrix
row = np.array([[1, 2, 3]])          # shape (1, 3) - a row matrix

print("vector:", vec.shape)
print("column:", col.shape)
print("row:   ", row.shape)


vector: (3,)
column: (3, 1)
row:    (1, 3)


### 2.5 If you ever need to change dtype

`.astype()` returns a converted copy. That's all you need for now: one function, used occasionally when a downstream function expects a specific dtype.


In [25]:
as_float = ints.astype(np.float64)
print(ints.dtype, "->", as_float.dtype)


int64 -> float64


> **ML tie-in:** a dataset is a `(n_samples, n_features)` array. Almost every confusing scikit-learn or PyTorch error you'll hit as a beginner is really a shape mismatch, so reading `.shape` fluently is one of the highest-leverage skills in this whole notebook.


## 3. Indexing, slicing, and views

NumPy indexing extends Python's list slicing to multiple dimensions, then adds boolean masking and fancy indexing. This is how you'll select feature columns, filter rows by a condition, or grab a mini-batch.


### 3.1 1-D indexing and slicing

Same rules as Python lists: zero-based, negative indices count from the end, `[start:stop]` excludes `stop`.


In [26]:
v = np.array([10, 20, 30, 40, 50])
print(v[0])
print(v[-1])
print(v[1:3])


10
50
[20 30]


### 3.2 2-D indexing: rows, columns, and sub-blocks

Use `a[row, col]` rather than chaining `a[row][col]`. It's the NumPy idiom.


In [27]:
m = np.arange(12).reshape(3, 4)
print(m)
print()
print("single element m[1, 2]:", m[1, 2])
print("whole row m[1, :]:     ", m[1, :])
print("whole column m[:, 2]:  ", m[:, 2])
print("sub-block m[0:2, 1:3]:\n", m[0:2, 1:3])


[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

single element m[1, 2]: 6
whole row m[1, :]:      [4 5 6 7]
whole column m[:, 2]:   [ 2  6 10]
sub-block m[0:2, 1:3]:
 [[1 2]
 [5 6]]


### 3.3 Boolean masking

A comparison on an array produces a boolean array of the same shape. Indexing with that boolean array keeps only the `True` positions. Combine conditions with `&` / `|` (with parentheses around each side), not Python's `and` / `or`, which don't work element-wise.


In [28]:
scores = np.array([55, 82, 91, 40, 67, 78])
passing = scores >= 60

print(passing)
print(scores[passing])
print(scores[(scores >= 60) & (scores < 90)])


[False  True  True False  True  True]
[82 91 67 78]
[82 67 78]


### 3.4 Fancy indexing

Indexing with a list/array of integers selects exactly those positions, in that order, repeats allowed.


In [29]:
data = np.array([10, 20, 30, 40, 50])
idx = [4, 0, 0, 2]
print(data[idx])


[50 10 10 30]


### 3.5 The views gotcha

Basic slicing shares memory with the original array: modifying the slice modifies the original too. This is a common surprise if you're used to Python list slicing, which always copies. When you need an independent array, call `.copy()` explicitly.


In [30]:
original = np.array([1, 2, 3, 4, 5])
view = original[1:4]
view[0] = 99
print("original changed too:", original)

safe_copy = np.array([1, 2, 3, 4, 5])[1:4].copy()
safe_copy[0] = -1
print("a .copy() stays independent:", safe_copy)


original changed too: [ 1 99  3  4  5]
a .copy() stays independent: [-1  3  4]


### 3.6 Pure Python vs NumPy: filtering

In plain Python, filtering a list means a list comprehension:


In [31]:
raw_scores = [55, 82, 91, 40, 67, 78]
passing_list = [s for s in raw_scores if s >= 60]
print(passing_list)


[82, 91, 67, 78]


With a NumPy array, the boolean mask from section 3.3 *is* the filter, no loop required:


In [32]:
print(scores[scores >= 60])


[82 91 67 78]


> **ML tie-in:** selecting feature columns, filtering rows by label, grabbing a mini-batch by index, and cleaning out invalid rows all lean on the indexing patterns from this chapter.


## 4. Reshaping, joining, and splitting arrays

Feeding data into a model almost always requires reshaping: flattening a batch of images into vectors, turning a prediction vector into a column, or assembling a feature matrix from pieces.


### 4.1 `.reshape()`: changing shape without changing data

The total number of elements must stay the same before and after. `-1` tells NumPy to infer that dimension for you.


![diagram](assets/numpy/diagram_reshape.png)


In [33]:
v = np.arange(6)
print(v.reshape(2, 3))
print(v.reshape(-1, 1))   # turn a flat vector into a column


[[0 1 2]
 [3 4 5]]
[[0]
 [1]
 [2]
 [3]
 [4]
 [5]]


### 4.2 `.ravel()` vs `.flatten()`: back to 1-D

Both flatten an array. `.ravel()` returns a view when possible; `.flatten()` always returns a copy. Prefer `.flatten()` when you need a guaranteed independent array.


In [34]:
grid = np.array([[1, 2], [3, 4]])
print(grid.ravel())
print(grid.flatten())


[1 2 3 4]
[1 2 3 4]


### 4.3 `.T`: transpose


![diagram](assets/numpy/diagram_transpose.png)


In [35]:
mat = np.array([[1, 2, 3], [4, 5, 6]])
print(mat.shape, "->", mat.T.shape)
print(mat.T)


(2, 3) -> (3, 2)
[[1 4]
 [2 5]
 [3 6]]


### 4.4 `np.newaxis` and `.squeeze()`: inserting and removing size-1 axes

A very common shape bug: you have a `(100,)` vector, but a function expects `(100, 1)`. `np.newaxis` fixes that without touching the data; `.squeeze()` reverses it.


![diagram](assets/numpy/diagram_newaxis.png)


In [36]:
vec = np.arange(5)
col = vec[:, np.newaxis]
print("vec shape:", vec.shape, "-> col shape:", col.shape)
print("squeezed back:", np.squeeze(col).shape)


vec shape: (5,) -> col shape: (5, 1)
squeezed back: (5,)


### 4.5 `np.concatenate` vs `np.stack`

`concatenate` joins arrays along an *existing* axis. `stack` joins them along a *brand-new* axis, useful for turning several individual samples into one batch. `np.hstack` / `np.vstack` are friendly shortcuts for the most common concatenation directions.


In [37]:
x1 = np.array([[1, 2], [3, 4]])
x2 = np.array([[5, 6]])
print("concatenate (existing axis):\n", np.concatenate([x1, x2], axis=0))

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("stack (new axis):\n", np.stack([a, b]))          # shape (2, 3)
print("vstack (same result here):\n", np.vstack([a, b]))


concatenate (existing axis):
 [[1 2]
 [3 4]
 [5 6]]
stack (new axis):
 [[1 2 3]
 [4 5 6]]
vstack (same result here):
 [[1 2 3]
 [4 5 6]]


### 4.6 `np.split`: the inverse of concatenation


In [38]:
whole = np.arange(9)
parts = np.split(whole, 3)
print(parts)


[array([0, 1, 2]), array([3, 4, 5]), array([6, 7, 8])]


### 4.7 A close cousin: `np.resize`

`.reshape()` requires the same total element count. `np.resize()` doesn't: a smaller target truncates, a larger one repeats the original data to fill the new shape. Reach for `.reshape()` by default, and only use `resize` when you deliberately want that repeating/truncating behavior.


In [39]:
small = np.arange(4)
print(np.resize(small, (2, 4)))   # repeats to fill a larger shape


[[0 1 2 3]
 [0 1 2 3]]


### 4.8 For the curious: why `reshape` doesn't copy

An array's shape is really just metadata (dtype, dimensions) describing how to read one flat block of memory. Reshaping changes that metadata, not the underlying bytes, which is why it's nearly free. This is the same block of memory read as a flat run and then as a 3x4 grid:

![NumPy array as metadata over a flat memory block](assets/numpy/numpy-array_memory.png)


> **ML tie-in:** flattening an image batch before a dense layer, turning a 1-D prediction array into a column, and assembling a feature matrix from separate arrays all use these tools.


## 5. Iterating arrays

We're about to show you the slow way on purpose. Most of the time, reaching for a loop over an array's elements means you actually want a vectorized operation instead, which is exactly Chapter 6. This chapter is the setup; the next one is the punchline.


### 5.1 A plain loop over a 1-D array


In [40]:
prices = np.array([10, 20, 30, 40, 50])
doubled = []
for p in prices:
    doubled.append(p * 2)
print(doubled)


[np.int64(20), np.int64(40), np.int64(60), np.int64(80), np.int64(100)]


### 5.2 It gets worse in 2-D (and worse still in 3-D)

Looping directly over a 2-D array yields whole rows, not individual elements, so reaching each number needs a second, nested loop. Every extra dimension costs you another level of nesting.


In [41]:
grid = np.array([[1, 2, 3], [4, 5, 6]])
doubled_grid = []
for row in grid:
    new_row = []
    for value in row:
        new_row.append(value * 2)
    doubled_grid.append(new_row)
print(doubled_grid)


[[np.int64(2), np.int64(4), np.int64(6)], [np.int64(8), np.int64(10), np.int64(12)]]


### 5.3 `np.nditer`: one flat loop, regardless of dimensions

If you truly must iterate, `np.nditer` walks every element in a single loop, no matter how many axes the array has.


In [42]:
for value in np.nditer(grid):
    print(value, end=" ")
print()


1 2 3 4 5 6 


### 5.4 `np.ndenumerate`: iterating with index positions

Like Python's `enumerate()`, but for arrays of any dimension: it yields `(index_tuple, value)` pairs.


In [43]:
for index, value in np.ndenumerate(grid):
    print(f"grid{index} = {value}")


grid(0, 0) = 1
grid(0, 1) = 2
grid(0, 2) = 3
grid(1, 0) = 4
grid(1, 1) = 5
grid(1, 2) = 6


## 6. Vectorization and broadcasting

**Vectorization** means expressing a computation as whole-array operations instead of a Python loop. **Broadcasting** is the set of rules NumPy uses to combine arrays of different but compatible shapes without you manually replicating data. Together, these are why NumPy code stays so compact, and they're the single most important idea in this notebook.


### 6.1 The payoff: replacing the loop from Chapter 5

Here's the exact nested loop from section 5.2, and the one line that replaces it:


In [44]:
grid = np.array([[1, 2, 3], [4, 5, 6]])
print(grid * 2)


[[ 2  4  6]
 [ 8 10 12]]


No loop, no nesting, and it works identically whether `grid` has 2 dimensions or 20. That's the whole argument for vectorization, in one line.


### 6.2 Element-wise operators and ufuncs

Every arithmetic operator applies element-wise. NumPy also ships vectorized math functions (*universal functions*, or ufuncs) like `sqrt`, `exp`, and `log` that behave the same way.


In [45]:
x = np.array([1, 4, 9, 16])
print(x + 10)
print(x > 5)
print(np.sqrt(x))


[11 14 19 26]
[False False  True  True]
[1. 2. 3. 4.]


### 6.3 Broadcasting with a scalar

You've already been doing this: `x + 10` "stretches" the scalar `10` to match every element of `x`. That's the simplest case of broadcasting.


### 6.4 The broadcasting rule

Compare shapes from the **right**. Two dimensions are compatible if they're equal, or if one of them is 1. If no dimension pair satisfies this, NumPy raises an error instead of guessing.


In [46]:
A = np.ones((3, 4))
b_row = np.array([1, 2, 3, 4])   # shape (4,) broadcasts against A's shape (3, 4)
print((A + b_row).shape)


(3, 4)


### 6.5 Broadcasting a row vector across every row

This is exactly how feature centering works: subtract a per-column mean (shape `(n_features,)`) from every row of a `(n_samples, n_features)` matrix.


![diagram](assets/numpy/diagram_broadcast_row.png)


In [47]:
X = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
col_mean = X.mean(axis=0)
print("column means:", col_mean)
print(X - col_mean)


column means: [ 2. 20.]
[[ -1. -10.]
 [  0.   0.]
 [  1.  10.]]


### 6.6 Broadcasting a column vector across every column

To broadcast *down* the rows instead, reshape the vector to a column `(n_samples, 1)`, exactly the `np.newaxis` trick from Chapter 4.


![diagram](assets/numpy/diagram_broadcast_col.png)


In [48]:
row_mean = X.mean(axis=1).reshape(-1, 1)   # shape (3, 1)
print(X - row_mean)


[[ -4.5   4.5]
 [ -9.    9. ]
 [-13.5  13.5]]


### 6.7 When broadcasting fails

If shapes truly aren't compatible, NumPy raises a `ValueError` immediately, a good thing, since it surfaces shape bugs early instead of silently producing nonsense.


In [49]:
try:
    np.array([1, 2, 3]) + np.array([1, 2])
except ValueError as e:
    print("ValueError:", e)


ValueError: operands could not be broadcast together with shapes (3,) (2,) 


> **ML tie-in:** feature centering, applying an activation function to a whole layer's output, and scaling every row of a dataset by a per-feature factor are all one broadcasted expression, no loop required.


## 7. Aggregations, sorting, and searching

Computing per-feature statistics, summarizing model outputs, and evaluating metrics all lean on aggregation functions. The `axis` parameter is the part people trip over most, so we build intuition for it slowly.


### 7.1 `.sum()` with no axis, then with one

With no axis, the whole array collapses to one number. `axis=0` collapses rows, leaving one value per column; `axis=1` collapses columns, leaving one value per row.


![diagram](assets/numpy/diagram_axis.png)


In [59]:
X = np.array([[1, 2, 3], [4, 5, 6]])
print("no axis: ", X.sum())
print("axis=0:  ", X.sum(axis=0), " (down each column)")
print("axis=1:  ", X.sum(axis=1), " (across each row)")


no axis:  21
axis=0:   [5 7 9]  (down each column)
axis=1:   [ 6 15]  (across each row)


### 7.2 `.mean()`, `.std()`, `.min()`, `.max()`: same `axis` logic


In [60]:
print("mean per column:", X.mean(axis=0))
print("std per column: ", X.std(axis=0))
print("max per row:    ", X.max(axis=1))


mean per column: [2.5 3.5 4.5]
std per column:  [1.5 1.5 1.5]
max per row:     [3 6]


### 7.3 `.argmin()` / `.argmax()`: the position of the extreme value

This is what you use to turn a model's per-class scores into a predicted class label.


In [61]:
class_scores = np.array([[0.1, 0.7, 0.2], [0.9, 0.05, 0.05]])
predicted_class = class_scores.argmax(axis=1)
print(predicted_class)


[1 0]


### 7.4 `keepdims=True`: keeping the axis for broadcasting afterward

By default, aggregating drops the axis you reduced over. `keepdims=True` keeps it as size 1, which lets the result broadcast cleanly back against the original array.


In [62]:
row_sums = X.sum(axis=1, keepdims=True)
print(row_sums.shape)   # (2, 1), not (2,)
print(X / row_sums)     # each row now sums to 1


(2, 1)
[[0.16666667 0.33333333 0.5       ]
 [0.26666667 0.33333333 0.4       ]]


### 7.5 `np.sort` and `np.argsort`

`sort` gives you the sorted values; `argsort` gives you the *indices* that would sort the array, often more useful since you can use it to reorder a second, related array.


In [63]:
unsorted = np.array([30, 10, 20])
print(np.sort(unsorted))
print(np.argsort(unsorted))


[10 20 30]
[1 2 0]


### 7.6 `np.unique`: distinct values and their counts

Handy for inspecting class labels in a dataset: how many distinct classes, and how many samples per class.


In [64]:
labels = np.array([0, 1, 1, 2, 2, 2, 0])
values, counts = np.unique(labels, return_counts=True)
print("classes:", values)
print("counts: ", counts)


classes: [0 1 2]
counts:  [2 2 3]


### 7.7 A light touch: `np.searchsorted`

On an already-sorted array, `searchsorted` finds where a value would need to be inserted to keep it sorted, in O(log n) instead of scanning linearly. One example is enough to recognize it later.


In [65]:
sorted_arr = np.array([1, 2, 3, 6, 7, 8, 9, 10])
print(np.searchsorted(sorted_arr, 4))


3


> **ML tie-in:** per-feature statistics for normalization, predicted class from a score matrix, and inspecting class balance with `unique` are daily-use patterns in a real ML pipeline.


## 8. Linear algebra essentials

Linear regression, PCA, and the forward pass of a neural network are all, at their core, matrix multiplications. We name the tools here and their ML role; we don't derive the math.


### 8.1 Elementwise `*` vs matrix multiplication `@`

This is one of the most common NumPy mistakes: `*` multiplies element-by-element; `@` (or `np.matmul`) does real matrix multiplication. Both produce a same-shaped result here, but with very different numbers, and no exception warns you if you use the wrong one.


In [66]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A * B (elementwise):\n", A * B)
print("A @ B (matrix multiply):\n", A @ B)


A * B (elementwise):
 [[ 5 12]
 [21 32]]
A @ B (matrix multiply):
 [[19 22]
 [43 50]]


### 8.2 Dot product, and matrix-vector multiplication as a linear layer

`weights @ x + bias` is exactly the computation inside a linear/dense layer. Shapes must line up: a `(m, n)` weight matrix times an `(n,)` input gives an `(m,)` output.


In [67]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print("dot product:", u @ v)

weights = np.array([[0.2, 0.8], [0.5, -0.5], [1.0, 1.0]])   # 3 outputs, 2 inputs
x = np.array([1.0, 2.0])
bias = np.array([0.1, 0.0, -0.1])
print("linear layer output:", weights @ x + bias)


dot product: 32
linear layer output: [ 1.9 -0.5  2.9]


### 8.3 Inverse, determinant, and `solve`

For solving a linear system `Ax = b`, `np.linalg.solve(A, b)` is both faster and more numerically stable than computing `inv(A) @ b` explicitly, so prefer it whenever you just need the solution.


In [68]:
print("inverse:\n", np.linalg.inv(A))
print("determinant:", np.linalg.det(A))

b_sys = np.array([9, 8])
A_sys = np.array([[3, 1], [1, 2]])
print("solve(A, b):", np.linalg.solve(A_sys, b_sys))


inverse:
 [[-2.   1. ]
 [ 1.5 -0.5]]
determinant: -2.0000000000000004
solve(A, b): [2. 3.]


### 8.4 A glance at `eig` and `svd`

Eigenvalues/eigenvectors and Singular Value Decomposition are the numerical machinery behind PCA (dimensionality reduction) later in the course. For now, just know the functions exist and what they're for.


In [69]:
eigvals, eigvecs = np.linalg.eig(A)
print("eigenvalues:", eigvals)


eigenvalues: [-0.37228132+0.j  5.37228132+0.j]


> **ML tie-in:** the forward pass of a neural network, linear regression's closed-form solution, and PCA all reduce to the handful of operations in this chapter.


## 9. Random number generation

Random splitting of data, shuffling, weight initialization, and synthetic noise all need random numbers, and reproducibility (the *same* random numbers on every run) matters for debugging and fair comparisons.


### 9.1 `np.random.default_rng`: the modern, recommended API

Passing a `seed` makes the sequence of "random" numbers fully reproducible: the same seed always produces the same output.


In [70]:
rng = np.random.default_rng(seed=42)
print(rng.integers(low=0, high=10, size=5))


[0 7 6 4 4]


### 9.2 The core sampling methods


In [71]:
print("uniform floats:", rng.random(size=3))
print("normal samples:", rng.normal(loc=0.0, scale=1.0, size=3))
print("choice (no replace):", rng.choice(np.arange(10), size=5, replace=False))


uniform floats: [0.69736803 0.09417735 0.97562235]
normal samples: [ 0.1278404  -0.31624259 -0.01680116]
choice (no replace): [9 6 3 4 5]


### 9.3 `.shuffle()` vs `.permutation()`

`.shuffle()` reorders an array in place; `.permutation()` returns a new shuffled array and leaves the input untouched.


In [72]:
deck = np.arange(10)
rng.shuffle(deck)
print("shuffled in place:", deck)

untouched = np.arange(10)
shuffled_copy = rng.permutation(untouched)
print("permutation copy:", shuffled_copy)
print("original untouched:", untouched)


shuffled in place: [1 0 5 4 8 9 7 2 3 6]
permutation copy: [2 8 9 7 4 1 6 3 5 0]
original untouched: [0 1 2 3 4 5 6 7 8 9]


### 9.4 Putting it together: a manual train/test split


In [73]:
n_samples = 10
indices = rng.permutation(n_samples)
split_point = int(0.8 * n_samples)
train_idx, test_idx = indices[:split_point], indices[split_point:]
print("train indices:", train_idx)
print("test indices: ", test_idx)


train indices: [6 3 8 2 9 5 1 7]
test indices:  [4 0]


### 9.5 A note on older code

You'll still see `np.random.seed(...)` with `np.random.rand()` / `randn()` / `randint()` in tutorials and older codebases. They work, but rely on shared global state; `default_rng` avoids that, so prefer it in your own code.


> **ML tie-in:** splitting data into train/test sets, shuffling before each training epoch, initializing weights, and adding synthetic noise all run through this chapter's tools.


## 10. Inserting, appending, and deleting elements

Datasets you load are usually fixed-size, but occasionally you need to grow or shrink an array directly, e.g. adding a computed feature column or dropping an outlier row. These functions all return a **new** array; none of them modify the original in place.


### 10.1 `np.insert` and `np.append`


In [74]:
base = np.arange(5)
print(np.insert(base, 1, 99))    # insert 99 at index 1
print(np.append(base, [100, 200]))


[ 0 99  1  2  3  4]
[  0   1   2   3   4 100 200]


### 10.2 `np.delete`


In [75]:
print(np.delete(base, 2))            # remove the element at index 2
print(np.delete(base, [0, 1]))       # remove several indices at once


[0 1 3 4]
[2 3 4]


> **Important warning:** every one of these allocates a brand-new array and copies everything into it. Calling `np.append` repeatedly inside a loop is quadratic, since it re-copies the growing array on every single call. We measure exactly how costly this is in the next chapter, and the fix (preallocating instead).


## 11. Performance: why vectorize?

This is where the memory and speed promises from the introduction get paid off in full, with real measurements on large arrays.


### 11.1 Memory: a list of a million integers vs an array

Recall the picture from the introduction: a Python list stores pointers to separately boxed integer objects, while a NumPy array stores raw numbers contiguously. Let's put numbers on that difference. `sys.getsizeof(py_list)` alone only counts the list's pointer array, not the integer objects each pointer refers to, so we add those in for the fair, full comparison.


In [76]:
import sys

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n, dtype=np.int64)

list_ptrs_only = sys.getsizeof(py_list)
one_int_object = sys.getsizeof(n)
list_full_cost = list_ptrs_only + one_int_object * n

print(f"list (pointers only):     {list_ptrs_only / 1e6:.1f} MB")
print(f"list (incl. int objects): {list_full_cost / 1e6:.1f} MB")
print(f"numpy array:              {np_arr.nbytes / 1e6:.1f} MB")


list (pointers only):     8.0 MB
list (incl. int objects): 36.0 MB
numpy array:              8.0 MB


### 11.2 Speed: one simple operation on ten million items

We measure a reduction (`sum`) rather than a mapping (like `* 2`), because a mapping has to allocate a brand-new ten-million-element output array, and that allocation cost shrinks the visible gap. A reduction produces a single number, so it isolates the raw compute speed and shows the advantage more cleanly. Both comparisons are honest; this one is just the clearer teaching example.


In [77]:
import time

n = 10_000_000
lst = list(range(n))
arr = np.arange(n)

start = time.perf_counter()
total_py = sum(lst)
py_time = time.perf_counter() - start

start = time.perf_counter()
total_np = arr.sum()
np_time = time.perf_counter() - start

print(f"pure Python: {py_time:.4f}s")
print(f"NumPy:       {np_time:.4f}s")
print(f"speedup:     {py_time / np_time:.0f}x")


pure Python: 0.0532s
NumPy:       0.0049s
speedup:     11x


### 11.3 Confirming the warning from Chapter 10: don't grow arrays in a loop


In [78]:
start = time.perf_counter()
grown = np.array([])
for i in range(5000):
    grown = np.append(grown, i)
append_time = time.perf_counter() - start

start = time.perf_counter()
preallocated = np.empty(5000)
for i in range(5000):
    preallocated[i] = i
prealloc_time = time.perf_counter() - start

print(f"repeated np.append: {append_time:.4f}s")
print(f"preallocated fill:  {prealloc_time:.4f}s")


repeated np.append: 0.0276s
preallocated fill:  0.0012s


### 11.4 A realistic example: mean squared error, loop vs vectorized


In [79]:
rng = np.random.default_rng(0)
y_true = rng.normal(size=100_000)
y_pred = y_true + rng.normal(scale=0.1, size=100_000)

start = time.perf_counter()
total = 0.0
for t, p in zip(y_true, y_pred):
    total += (t - p) ** 2
mse_loop = total / len(y_true)
loop_time = time.perf_counter() - start

start = time.perf_counter()
mse_vector = np.mean((y_true - y_pred) ** 2)
vector_time = time.perf_counter() - start

print(f"MSE (loop):       {mse_loop:.6f}  in {loop_time:.4f}s")
print(f"MSE (vectorized): {mse_vector:.6f}  in {vector_time:.4f}s")
print(f"speedup: {loop_time / vector_time:.1f}x")


MSE (loop):       0.010045  in 0.0513s
MSE (vectorized): 0.010045  in 0.0005s
speedup: 111.0x


> **ML tie-in:** this is why training and preprocessing at real dataset sizes is feasible at all. Every chapter before this one has been building toward this exact payoff.


## 12. Summary

### Cheat sheet

| Task | Function |
|---|---|
| Create an array | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.eye`, `np.arange`, `np.linspace` |
| Inspect an array | `.shape`, `.ndim`, `.size`, `.dtype` |
| Select data | slicing `a[i:j]`, boolean masks `a[a > x]`, fancy indexing `a[[i, j, k]]` |
| Reshape / combine | `.reshape()`, `.ravel()`/`.flatten()`, `.T`, `np.newaxis`, `.squeeze()`, `np.concatenate`, `np.stack`, `np.split` |
| Iterate (rarely) | `np.nditer()`, `np.ndenumerate()` |
| Elementwise math | `+ - * / **`, ufuncs (`np.sqrt`, `np.exp`, `np.log`) |
| Matrix math | `@` / `np.matmul`, `np.linalg.inv`, `np.linalg.solve`, `np.linalg.eig` |
| Aggregate / sort / search | `.sum()/.mean()/.std()`, `.argmin()/.argmax()`, `keepdims=True`, `np.sort`/`np.argsort`, `np.unique` (always mind `axis`) |
| Grow / shrink | `np.insert`, `np.append`, `np.delete` (all return new arrays; never in a loop) |
| Random | `np.random.default_rng(seed=...)`, `.integers`/`.random`/`.normal`/`.choice`/`.shuffle`/`.permutation` |


## 13. Exercises

1. Create a `(5, 5)` array of random integers between 0 and 100. Replace every value greater than 50 with 0, using boolean masking (no loops).
2. Given a `(10, 3)` array representing 10 samples with 3 features, compute the min-max normalized version (each feature scaled to `[0, 1]`) using only vectorized operations and broadcasting.
3. Write a vectorized version of the Euclidean distance between two `(n,)` vectors, and confirm it matches `np.linalg.norm(a - b)`.
4. Time a vectorized computation of `x**2 + 2*x + 1` for 1 million elements against the equivalent Python `for` loop, and report the speedup yourself.
5. Given a `(10, 3)` array of class scores (one row per sample, one column per class) and a `(10,)` array of true labels, compute the predicted label for each sample and the overall accuracy, without writing a loop.
